In [ ]:
  # Paste this entire cell into a Kaggle notebook after attaching:
  # 1) competition data, and 2) E70a2 reduced-Ridge bundle.
  from pathlib import Path
  import hashlib
  import subprocess
  import sys
  import zipfile

  import numpy as np
  import pandas as pd

  INPUT = Path('/kaggle/input')
  WORKING = Path('/kaggle/working')
  OUTPUT = WORKING / 'submission.csv'

  def competition_root(path: Path) -> bool:
      return (path / 'sample_submission.csv').is_file() and (path / 'test').is_dir()

  data_candidates = [p for p in INPUT.rglob('*') if p.is_dir() and competition_root(p)]
  if not data_candidates:
      raise FileNotFoundError('Competition data not found below /kaggle/input')
  DATA = min(data_candidates, key=lambda p: len(p.parts))

  launchers = list(INPUT.rglob('run_kaggle_e70a2.py'))
  if not launchers:
      archives = list(INPUT.rglob('*e70a2*bundle.zip'))
      if len(archives) != 1:
          raise RuntimeError(
              f'Expected one unpacked bundle or one E70a2 ZIP, '
              f'found {len(launchers)} launchers and {len(archives)} ZIPs'
          )
      unpack = WORKING / 'e70a2_bundle_unpacked'
      with zipfile.ZipFile(archives[0]) as archive:
          members = archive.namelist()
          if any(
              Path(member).is_absolute() or '..' in Path(member).parts
              for member in members
          ):
              raise RuntimeError('Unsafe ZIP member')
          archive.extractall(unpack)
      launchers = list(unpack.rglob('run_kaggle_e70a2.py'))

  if len(launchers) != 1:
      raise RuntimeError(
          f'Expected exactly one E70a2 launcher after discovery, '
          f'found {len(launchers)}'
      )

  BUNDLE = launchers[0].parent

  print('Competition data:', DATA)
  print('E70a2 bundle:', BUNDLE)

  subprocess.run([
      sys.executable, '-u', str(BUNDLE / 'run_kaggle_e70a2.py'),
      '--data-dir', str(DATA),
      '--output', str(OUTPUT),
      '--work-dir', str(WORKING / 'e70a2_runtime'),
      '--device', 'cuda:0',
      '--jobs', '4',
      '--well-chunk-size', '4',
  ], check=True)

  sample = pd.read_csv(DATA / 'sample_submission.csv')
  submission = pd.read_csv(OUTPUT)

  if list(submission.columns) != list(sample.columns):
      raise RuntimeError(f'Unexpected columns: {submission.columns.tolist()}')
  if not submission.id.astype(str).equals(sample.id.astype(str)):
      raise RuntimeError('Submission IDs/order differ from sample_submission.csv')
  if submission.id.duplicated().any() or not np.isfinite(submission.tvt.to_numpy(float)).all():
      raise RuntimeError('Duplicate IDs or non-finite predictions')

  print(f'READY: {OUTPUT}')
  print(f'rows={len(submission)}, sha256={hashlib.sha256(OUTPUT.read_bytes()).hexdigest()}')
  display(submission.head())

In [ ]:
  import os
  import shutil
  import subprocess
  import sys
  from pathlib import Path

  INPUT = Path("/kaggle/input")
  WORK = Path("/kaggle/working")

  DATA = Path(
      "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
  )

  UPLOADED_CODE = Path(
      "/kaggle/input/datasets/lihu6666/input-qianzhui"
  )

  RAW_PHYSICS_RUNNER = Path(
      "/kaggle/input/datasets/lihu6666/ridge/"
      "kaggle_e70a2_reduced_ridge_full773_20260801_v2/"
      "runtime/physics/run_kaggle.py"
  )

  MODEL_DIR = Path(
      "/kaggle/input/datasets/lihu6666/ridge/"
      "kaggle_e70a2_reduced_ridge_full773_20260801_v2/"
      "models/physics_models"
  )

  # ------------------------------------------------------------
  # 1. 将上传的 Prefix Gold 脚本复制到可写目录，并修复 row_idx 兼容性
  # ------------------------------------------------------------

  CODE = WORK / "prefix_gold_code"
  CODE.mkdir(parents=True, exist_ok=True)

  for name in [
      "prefix_gold_candidate.py",
      "run_kaggle_prefix_gold.py",
  ]:
      shutil.copy2(UPLOADED_CODE / name, CODE / name)

  runner_code = CODE / "run_kaggle_prefix_gold.py"

  text = runner_code.read_text(encoding="utf-8")

  old = 'all_rows = horizontal.row_idx.to_numpy(dtype=int)[cutoff + 1:]'
  new = 'all_rows = row_ids(horizontal)[cutoff + 1:]'

  if old in text:
      text = text.replace(old, new)

  runner_code.write_text(text, encoding="utf-8")

  assert "all_rows = row_ids(horizontal)[cutoff + 1:]" in (
      runner_code.read_text(encoding="utf-8")
  )

  # ------------------------------------------------------------
  # 2. 冻结安全 Ridge8 基线
  # ------------------------------------------------------------

  BASE = WORK / "submission_safe_ridge8.csv"

  if not BASE.is_file():
      shutil.copy2(WORK / "submission.csv", BASE)

  physics_outputs = [
      p for p in WORK.rglob("physics_submission.csv")
      if "prefix_gold" not in p.parts
  ]

  assert len(physics_outputs) == 1, physics_outputs

  PHYSICS_SUBMISSION = physics_outputs[0]

  # ------------------------------------------------------------
  # 3. 创建可运行的 Physics runtime，并挂载同源模型权重
  # ------------------------------------------------------------

  required = [
      "tcn_fold0_real.pt",
      "ridge_fold0.joblib",
      "gru_fold0_real.pt",
      "point_rate_seed0_fold0.pt",
      "point_rate_seed2_fold4.pt",
  ]

  for name in required:
      assert (MODEL_DIR / name).is_file(), MODEL_DIR / name

  STAGE = WORK / "prefix_gold_physics_runtime"

  if not STAGE.exists():
      shutil.copytree(
          RAW_PHYSICS_RUNNER.parent,
          STAGE,
          symlinks=True,
      )

  models_link = STAGE / "models"

  if models_link.exists() or models_link.is_symlink():
      if models_link.is_dir() and not models_link.is_symlink():
          shutil.rmtree(models_link)
      else:
          models_link.unlink()

  os.symlink(MODEL_DIR, models_link, target_is_directory=True)

  PHYSICS_RUNNER = STAGE / "run_kaggle.py"

  assert PHYSICS_RUNNER.is_file()
  assert (
      PHYSICS_RUNNER.parent / "models" / "point_rate_seed0_fold0.pt"
  ).is_file()

  # ------------------------------------------------------------
  # 4. 测试井 TVT_input 内部伪切分候选后处理
  # ------------------------------------------------------------

  PREFIX_WORK = WORK / "prefix_gold"

  print("BASE:", BASE)
  print("PHYSICS_SUBMISSION:", PHYSICS_SUBMISSION)
  print("PHYSICS_RUNNER:", PHYSICS_RUNNER)
  print("PATCHED_CODE:", runner_code)

  subprocess.run(
      [
          sys.executable,
          "-u",
          str(runner_code),
          "--data-dir",
          str(DATA),
          "--base-submission",
          str(BASE),
          "--physics-submission",
          str(PHYSICS_SUBMISSION),
          "--physics-runner",
          str(PHYSICS_RUNNER),
          "--output",
          str(WORK / "submission.csv"),
          "--work-dir",
          str(PREFIX_WORK),
          "--max-alpha",
            "0.30",
      ],
      check=True,
  )

  print("READY:", WORK / "submission.csv")
  print("SAFE BACKUP:", BASE)
  print("AUDIT:", PREFIX_WORK / "prefix_gold_test_audit.json")

In [ ]:

  # from pathlib import Path
  # import hashlib
  # import os
  # import shutil
  # import subprocess
  # import sys

  # import numpy as np
  # import pandas as pd

  # INPUT = Path("/kaggle/input")
  # WORKING = Path("/kaggle/working")
  # OUTPUT = WORKING / "submission.csv"

  # def is_competition_root(path: Path) -> bool:
  #     return (path / "sample_submission.csv").is_file() and (path / "test").is_dir()

  # data_roots = [
  #     path for path in INPUT.rglob("*")
  #     if path.is_dir() and is_competition_root(path)
  # ]
  # if not data_roots:
  #     raise FileNotFoundError("Competition data not found")
  # DATA = min(data_roots, key=lambda path: len(path.parts))

  # launchers = list(INPUT.rglob("run_kaggle_affine5.py"))
  # if len(launchers) != 1:
  #     raise RuntimeError(f"Expected one affine5 bundle, found {len(launchers)}")
  # BUNDLE = launchers[0].parent

  # # Kaggle Input 是只读：建立一个只复制代码、模型仍软链接回 Input 的热修复目录。
  # PATCH_ROOT = WORKING / "affine5_hotfix"
  # shutil.rmtree(PATCH_ROOT, ignore_errors=True)
  # PATCH_ROOT.mkdir(parents=True)

  # for name in ("runtime", "models", "meta", "affine_e54"):
  #     os.symlink(
  #         BUNDLE / name,
  #         PATCH_ROOT / name,
  #         target_is_directory=True,
  #     )

  # source = (BUNDLE / "run_kaggle_affine5.py").read_text(encoding="utf-8")

  # # 修复 Physics 子运行器寻找 runtime/physics/models 的旧路径问题。
  # old_stage = '''    work.mkdir(parents=True)
  #     physics_output = work / "physics_submission.csv"'''

  # new_stage = '''    work.mkdir(parents=True)
  #     physics_runtime = work / "code_runtime" / "physics"
  #     physics_runtime.parent.mkdir(parents=True)
  #     shutil.copytree(root / "runtime/physics", physics_runtime)
  #     os.symlink(
  #         root / "models/physics_models",
  #         physics_runtime / "models",
  #         target_is_directory=True,
  #     )
  #     physics_output = work / "physics_submission.csv"'''

  # old_call = '''sys.executable, "-u", root / "runtime/physics/run_kaggle.py",'''
  # new_call = '''sys.executable, "-u", physics_runtime / "run_kaggle.py",'''

  # if old_stage not in source or old_call not in source:
  #     raise RuntimeError("Hotfix target text not found; stop rather than patching an unknown launcher.")

  # source = source.replace(old_stage, new_stage, 1)
  # source = source.replace(old_call, new_call, 1)

  # HOTFIX_RUNNER = PATCH_ROOT / "run_kaggle_affine5.py"
  # HOTFIX_RUNNER.write_text(source, encoding="utf-8")

  # print("Competition data:", DATA)
  # print("Original bundle:", BUNDLE)
  # print("Patched launcher:", HOTFIX_RUNNER)

  # subprocess.run([
  #     sys.executable, "-u", str(HOTFIX_RUNNER),
  #     "--data-dir", str(DATA),
  #     "--output", str(OUTPUT),
  #     "--work-dir", str(WORKING / "affine5_runtime"),
  #     "--device", "cuda:0",
  #     "--jobs", "4",
  #     "--pf-jobs", "4",
  #     "--well-chunk-size", "4",
  # ], check=True)

  # sample = pd.read_csv(DATA / "sample_submission.csv")
  # submission = pd.read_csv(OUTPUT)

  # assert list(submission.columns) == list(sample.columns)
  # assert submission["id"].astype(str).equals(sample["id"].astype(str))
  # assert not submission["id"].duplicated().any()
  # assert np.isfinite(submission["tvt"].to_numpy(float)).all()

  # print("READY:", OUTPUT)
  # print("rows:", len(submission))
  # print("sha256:", hashlib.sha256(OUTPUT.read_bytes()).hexdigest())
  # display(submission.head())